In [1]:
!git clone https://github.com/pyther-hub/ml4sci-taylor-series-task.git

Cloning into 'ml4sci-taylor-series-task'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 44 (delta 19), reused 33 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 84.49 KiB | 5.28 MiB/s, done.
Resolving deltas: 100% (19/19), done.


In [2]:
%cd /kaggle/working/ml4sci-taylor-series-task

/kaggle/working/ml4sci-taylor-series-task


In [3]:
"""
main.py
==================
Training entry point for the unified Taylor coefficient prediction pipeline.

The model predicts ALL five Taylor coefficients in a single pass, using
<BREAK> tokens to delimit individual coefficients in the output sequence.

Training loop only computes teacher-forced val loss (fast).  After training,
evaluate_checkpoints() loads each saved epoch checkpoint, runs greedy decoding,
and computes full accuracy metrics.

Usage
-----
python main.py

Edit the hyperparameters in the ``if __name__ == "__main__":`` block at the
bottom of this file.
"""

from __future__ import annotations

import os
import time
from typing import Dict

import torch
import torch.nn as nn

from dataset import BREAK_ID, EOS_ID, N_COEFFS, PAD_ID, SOS_ID, VOCAB_SIZE, decode, encode
from model import CoeffPredTransformer
from train_validate import (
build_dataloaders,
get_device,
print_epoch,
set_seed,
train_epoch,
validate,
)

# ── Evaluation on fixed function list ─────────────────────────────────────
import sympy as sp
from dataset_generation import (
    _expr_to_prefix_tokens,
    compute_taylor_coefficients,
    prefix_tokens_to_infix,
)
from metrics import split_segments

EVAL_FUNCTIONS = [
    "(x**2 + 1)*sin(x)",
    "x**3*cos(2*x)",
    "(2*x + 1)*sin(3*x)",
    "(x**2 - x)*cos(x)",
    "exp(x)*(1 + x)",
    "x*exp(2*x)",
    "(1 + x**2)*exp(-x)",
    "exp(x)*sin(x)",
    "log(1 + x)",
    "log(1 + x**2)",
    "x*log(1 + x)",
    "log(1 + 2*x + x**2)",
    "x/(1 - x)",
    "x/(1 + x)",
    "1/(1 + x**2)",
    "x**2/(1 - x)",
    "sin(x**2)",
    "exp(x**2)",
    "cos(sqrt(1 + x))",
    "log(1 + sin(x))",
    "(x + 1)*exp(x)",
    "sin(x)/(1 + x)",
    "x**2*log(1 + x)",
    "exp(x)*cos(x)",
    "(x**2 + 2*x + 1)*sin(x)",
    "(x + 2)*cos(2*x)",
    "exp(x)*(x**2 + 1)",
    "(x**2 + 1)*exp(2*x)",
    "sin(x)*cos(x)",
    "sin(2*x)/(1 + x)",
    "cos(x)/(1 + x**2)",
    "(x + 1)/(1 + x**2)",
    "(x**2 + x)/(1 + x)",
    "(x**2 + 1)/(1 + x)",
    "log(1 + x + x**2)",
    "log(1 + x**3)",
    "log(1 + x)*sin(x)",
    "log(1 + x)*cos(x)",
    "x*exp(x)*sin(x)",
    "x*exp(x)*cos(x)",
    "exp(x)*log(1 + x)",
    "exp(x)*sqrt(1 + x)",
    "sqrt(1 + x)*sin(x)",
    "sqrt(1 + x)*cos(x)",
    "sqrt(1 + x)/(1 + x)",
    "sin(x)/(1 + x**2)",
    "cos(x)/(1 + x)",
    "(x + 1)*sin(x)*cos(x)",
    "(x**2 + 1)*sin(2*x)",
    "(x**2 + x + 1)*cos(x)",
    "(x + 1)*log(1 + x)",
    "(x**2 + 1)*log(1 + x)",
    "exp(x)/(1 + x)",
    "exp(x)/(1 + x**2)",
    "sin(x**2)/(1 + x)",
    "cos(x**2)/(1 + x)",
    "exp(x**2)/(1 + x)",
    "log(1 + x**2)*sin(x)",
    "log(1 + x**2)*cos(x)",
    "sin(x)*exp(x**2)",
    "cos(x)*exp(x**2)",
]

TAYLOR_ORDER = N_COEFFS - 1   # 4 → coefficients c0 … c4
x_sym        = sp.Symbol("x")


In [4]:

# ══════════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def build_model(
    d_model:            int,
    nhead:              int,
    num_encoder_layers: int,
    num_decoder_layers: int,
    dim_feedforward:    int,
    dropout:            float,
    max_seq_len:        int,
) -> CoeffPredTransformer:
    return CoeffPredTransformer(
        d_model=d_model,
        nhead=nhead,
        num_encoder_layers=num_encoder_layers,
        num_decoder_layers=num_decoder_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
        max_seq_len=max_seq_len,
    )


def save_checkpoint(
    model:       CoeffPredTransformer,
    path:        str,
    epoch:       int,
    val_metrics: Dict,
    arch_cfg:    Dict,
) -> None:
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    torch.save(
        {
            "epoch":       epoch,
            "n_coeffs":    N_COEFFS,
            "val_loss":    val_metrics.get("val_loss", float("inf")),
            "model_state": model.state_dict(),
            "config":      arch_cfg,
        },
        path,
    )
    print(f"  [save] checkpoint → {path}  (epoch={epoch})")



In [5]:
# ── Paths ─────────────────────────────────────────────────────────────────
DATASET_JSON    = os.path.join("", "/kaggle/input/datasets/tensorpanda231/taylor-series-dataset-simple/taylor_dataset_10k.json")
CHECKPOINT_DIR  = "checkpoints"
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "taylor_series_pred_baseline.pt")


# ── Data split ────────────────────────────────────────────────────────────
VAL_RATIO   = 0.10
RANDOM_SEED = 42

# ── Training ──────────────────────────────────────────────────────────────
BATCH_SIZE  = 64
NUM_EPOCHS  = 100
LR          = 3e-4
CLIP_GRAD   = 1.0
NUM_WORKERS = 0

# ── Model architecture ────────────────────────────────────────────────────
D_MODEL            = 256
NHEAD              = 8      # D_MODEL must be divisible by NHEAD
NUM_ENCODER_LAYERS = 6
NUM_DECODER_LAYERS = 8
DIM_FEEDFORWARD    = 256
DROPOUT            = 0.1
MAX_SEQ_LEN        = 512

# ── Post-training evaluation ──────────────────────────────────────────────
MAX_GEN_LEN = 512          # max decode steps for post-training greedy eval


In [6]:

# ─────────────────────────────────────────────────────────────────────────
set_seed(RANDOM_SEED)
device = get_device()
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("=" * 68)
print(f"  coeff_pred training — unified model ({N_COEFFS} coefficients)")
print(f"  device  : {device}")
print(f"  dataset : {DATASET_JSON}")
print("=" * 68)

# ── Data ──────────────────────────────────────────────────────────────────
train_loader, val_loader, full_ds = build_dataloaders(
    DATASET_JSON, VAL_RATIO, RANDOM_SEED, BATCH_SIZE, NUM_WORKERS,
    max_seq_len=MAX_SEQ_LEN,
)
print(f"\n  Dataset  : {len(full_ds)} items  (skipped={full_ds.n_skipped})")
print(f"  Train    : {len(train_loader.dataset)}   Val : {len(val_loader.dataset)}\n")

# ── Model ─────────────────────────────────────────────────────────────────
arch_cfg = dict(
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_seq_len=MAX_SEQ_LEN,
)
model    = build_model(**arch_cfg).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Params   : {n_params:,}   VOCAB_SIZE={VOCAB_SIZE}\n")

# ── Optimizer / loss / scheduler ──────────────────────────────────────────
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, reduction="mean")
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS, eta_min=LR * 1e-2,
)

# ── Training loop (no greedy decode — fast) ──────────────────────────────
best_val_loss = float("inf")


  coeff_pred training — unified model (5 coefficients)
  device  : cuda
  dataset : /kaggle/input/datasets/tensorpanda231/taylor-series-dataset-simple/taylor_dataset_10k.json

  Dataset  : 9944 items  (skipped=56)
  Train    : 8950   Val : 994

  Params   : 7,657,728   VOCAB_SIZE=29



In [7]:

for epoch in range(1, NUM_EPOCHS + 1):
    t0      = time.perf_counter()
    train_m = train_epoch(model, train_loader, optimizer, criterion, device, CLIP_GRAD)
    val_m   = validate(model, val_loader, criterion, device)
    scheduler.step()

    elapsed = time.perf_counter() - t0
    is_best = val_m["val_loss"] < best_val_loss

    # Save checkpoint every epoch (for post-training evaluation).
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch_{epoch:03d}.pt")
    torch.save(
        {
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "val_loss":    val_m["val_loss"],
            "n_coeffs":    N_COEFFS,
            "config":      arch_cfg,
        },
        ckpt_path,
    )

    if is_best:
        best_val_loss = val_m["val_loss"]
        save_checkpoint(model, CHECKPOINT_PATH, epoch, val_m, arch_cfg)

    print_epoch(epoch, NUM_EPOCHS, train_m, val_m, elapsed, is_best)

    if epoch%5 == 0:

        print("\n" + "=" * 68)
        print("  Evaluation on fixed function list (autoregressive decoding)")
        print("=" * 68)
        
        n_fn_correct  = 0
        n_fn_attempted = 0
        
        for fn_idx, fn_str in enumerate(EVAL_FUNCTIONS, 1):
        
            # 1. Parse string → sympy
            try:
                expr = sp.sympify(fn_str, locals={"x": x_sym})
            except Exception as e:
                print(f"\n  [{fn_idx:2d}] {fn_str}")
                print(f"       PARSE ERROR: {e}")
                continue
        
            # 2. Ground-truth Taylor coefficients (nth derivative at x=a)
            gt_coeffs = compute_taylor_coefficients(expr, TAYLOR_ORDER)
            if gt_coeffs is None:
                print(f"\n  [{fn_idx:2d}] {fn_str}")
                print(f"       COEFF ERROR: sympy timed out")
                continue
        
            gt_token_lists = [_expr_to_prefix_tokens(c) for c in gt_coeffs]
        
            # 3. Encode source sequence
            fn_prefix = _expr_to_prefix_tokens(expr)
            src_ids   = [SOS_ID] + encode(fn_prefix) + [EOS_ID]
            src       = torch.tensor([src_ids], dtype=torch.long).to(device)
        
            # 4. Autoregressive decoding — one token at a time, predicted token
            #    fed back as next input (implemented inside model.generate)
            pred_ids = model.generate(src, max_len=MAX_GEN_LEN)   # List[int]
        
            # 5. Split flat prediction on <BREAK> into per-coefficient segments
            pred_segs = split_segments(pred_ids, BREAK_ID, EOS_ID, PAD_ID)
        
            # 6. Compare and display
            n_fn_attempted += 1
            n_coeff_correct = 0
        
            print(f"\n  [{fn_idx:2d}] f(x) = {fn_str}")
            for coeff_i in range(N_COEFFS):
                gt_toks   = gt_token_lists[coeff_i] if coeff_i < len(gt_token_lists) else []
                pred_toks = pred_segs[coeff_i]       if coeff_i < len(pred_segs)     else []
        
                gt_ids    = encode(gt_toks)           # List[int]
                match     = (gt_ids == pred_toks)
        
                gt_infix   = prefix_tokens_to_infix(gt_toks)
                pred_infix = prefix_tokens_to_infix(decode(pred_toks, skip_special=False))
        
                status = "OK" if match else "--"
                print(
                    f"       c{coeff_i} [{status}]"
                    f"  gt={gt_infix}"
                    f"  pred={pred_infix}"
                )
                if match:
                    n_coeff_correct += 1
        
            all_correct = (n_coeff_correct == N_COEFFS)
            if all_correct:
                n_fn_correct += 1
            print(f"       => {n_coeff_correct}/{N_COEFFS} coefficients correct")
        
        print("\n" + "=" * 68)
        print(
            f"  Result : {n_fn_correct}/{n_fn_attempted} functions"
            f" with all {N_COEFFS} coefficients correct"
            f"  ({100*n_fn_correct/n_fn_attempted:.1f}%)" if n_fn_attempted else ""
        )
        print("=" * 68)

        

print(f"\n  Training complete.  Best val loss: {best_val_loss:.6f}")
print(f"  Best checkpoint: {CHECKPOINT_PATH}")

# ── Sample predictions from best checkpoint ──────────────────────────────
print("\n  Loading best model for example predictions ...")
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True)
model.load_state_dict(ckpt["model_state"])
model.eval()

print(f"\n  Example predictions (unified, greedy):")
shown = 0
with torch.no_grad():
    for src, tgt, _, _ in val_loader:
        src   = src.to(device)
        preds = model.generate_batch(src, max_len=MAX_GEN_LEN)
        for i in range(len(preds)):
            if shown >= 4:
                break
            src_tok  = decode(src[i].tolist())
            tgt_tok  = decode(tgt[i].tolist(), skip_special=False)
            pred_tok = decode([t for t in preds[i] if t != EOS_ID], skip_special=False)
            print(f"  ── example {shown + 1} ──")
            print(f"     src  : {src_tok}")
            print(f"     tgt  : {tgt_tok}")
            print(f"     pred : {pred_tok}")
            print(f"     match: {'✓' if tgt_tok == pred_tok else '✗'}")
            shown += 1
        if shown >= 4:
            break


  [save] checkpoint → checkpoints/taylor_series_pred_baseline.pt  (epoch=1)
  epoch   1/100  trn_loss=3.0748  val_loss=1.3641  val_tok=0.510  val_sent=0.007  94.7s  *best*
  [save] checkpoint → checkpoints/taylor_series_pred_baseline.pt  (epoch=2)
  epoch   2/100  trn_loss=1.3241  val_loss=1.0994  val_tok=0.602  val_sent=0.240  93.0s  *best*
  [save] checkpoint → checkpoints/taylor_series_pred_baseline.pt  (epoch=3)
  epoch   3/100  trn_loss=1.1025  val_loss=0.9094  val_tok=0.668  val_sent=0.293  93.3s  *best*
  [save] checkpoint → checkpoints/taylor_series_pred_baseline.pt  (epoch=4)
  epoch   4/100  trn_loss=0.9369  val_loss=0.7542  val_tok=0.727  val_sent=0.361  93.7s  *best*
  [save] checkpoint → checkpoints/taylor_series_pred_baseline.pt  (epoch=5)
  epoch   5/100  trn_loss=0.8107  val_loss=0.6585  val_tok=0.760  val_sent=0.394  93.5s  *best*

  Evaluation on fixed function list (autoregressive decoding)

  [ 1] f(x) = (x**2 + 1)*sin(x)
       c0 [--]  gt=a**2*sin(a) + sin(a)  pre

In [ ]:

# ── Evaluation on fixed function list ─────────────────────────────────────
import sympy as sp
from dataset_generation import (
    _expr_to_prefix_tokens,
    compute_taylor_coefficients,
    prefix_tokens_to_infix,
)
from metrics import split_segments


TAYLOR_ORDER = N_COEFFS - 1   # 4 → coefficients c0 … c4
x_sym        = sp.Symbol("x")

print("\n" + "=" * 68)
print("  Evaluation on fixed function list (autoregressive decoding)")
print("=" * 68)

n_fn_correct  = 0
n_fn_attempted = 0

for fn_idx, fn_str in enumerate(EVAL_FUNCTIONS, 1):

    # 1. Parse string → sympy
    try:
        expr = sp.sympify(fn_str, locals={"x": x_sym})
    except Exception as e:
        print(f"\n  [{fn_idx:2d}] {fn_str}")
        print(f"       PARSE ERROR: {e}")
        continue

    # 2. Ground-truth Taylor coefficients (nth derivative at x=a)
    gt_coeffs = compute_taylor_coefficients(expr, TAYLOR_ORDER)
    if gt_coeffs is None:
        print(f"\n  [{fn_idx:2d}] {fn_str}")
        print(f"       COEFF ERROR: sympy timed out")
        continue

    gt_token_lists = [_expr_to_prefix_tokens(c) for c in gt_coeffs]

    # 3. Encode source sequence
    fn_prefix = _expr_to_prefix_tokens(expr)
    src_ids   = [SOS_ID] + encode(fn_prefix) + [EOS_ID]
    src       = torch.tensor([src_ids], dtype=torch.long).to(device)

    # 4. Autoregressive decoding — one token at a time, predicted token
    #    fed back as next input (implemented inside model.generate)
    pred_ids = model.generate(src, max_len=MAX_GEN_LEN)   # List[int]

    # 5. Split flat prediction on <BREAK> into per-coefficient segments
    pred_segs = split_segments(pred_ids, BREAK_ID, EOS_ID, PAD_ID)

    # 6. Compare and display
    n_fn_attempted += 1
    n_coeff_correct = 0

    print(f"\n  [{fn_idx:2d}] f(x) = {fn_str}")
    for coeff_i in range(N_COEFFS):
        gt_toks   = gt_token_lists[coeff_i] if coeff_i < len(gt_token_lists) else []
        pred_toks = pred_segs[coeff_i]       if coeff_i < len(pred_segs)     else []

        gt_ids    = encode(gt_toks)           # List[int]
        match     = (gt_ids == pred_toks)

        gt_infix   = prefix_tokens_to_infix(gt_toks)
        pred_infix = prefix_tokens_to_infix(decode(pred_toks, skip_special=False))

        status = "OK" if match else "--"
        print(
            f"       c{coeff_i} [{status}]"
            f"  gt={gt_infix}"
            f"  pred={pred_infix}"
        )
        if match:
            n_coeff_correct += 1

    all_correct = (n_coeff_correct == N_COEFFS)
    if all_correct:
        n_fn_correct += 1
    print(f"       => {n_coeff_correct}/{N_COEFFS} coefficients correct")

print("\n" + "=" * 68)
print(
    f"  Result : {n_fn_correct}/{n_fn_attempted} functions"
    f" with all {N_COEFFS} coefficients correct"
    f"  ({100*n_fn_correct/n_fn_attempted:.1f}%)" if n_fn_attempted else ""
)
print("=" * 68)



  Evaluation on fixed function list (autoregressive decoding)

  [ 1] f(x) = (x**2 + 1)*sin(x)
       c0 [OK]  gt=a**2*sin(a) + sin(a)  pred=a**2*sin(a) + sin(a)
       c1 [--]  gt=a**2*cos(a) + 2*a*sin(a) + cos(a)  pred=a**2*cos(a) + 2*a*sin(a) + sin(a)
       c2 [--]  gt=-a**2*sin(a) + 4*a*cos(a) + sin(a)  pred=-a**2*sin(a) - 4*a*cos(a) + 2*sin(a) + 2*cos(a)
       c3 [--]  gt=-a**2*cos(a) - 6*a*sin(a) + 5*cos(a)  pred=-a**2*cos(a) - 12*a*sin(a) - 6*a*cos(a) - 6*cos(a)
       c4 [--]  gt=a**2*sin(a) - 8*a*cos(a) - 11*sin(a)  pred=a**2*sin(a) + 4*a*cos(a) - 12*sin(a)
       => 1/5 coefficients correct

  [ 2] f(x) = x**3*cos(2*x)
       c0 [OK]  gt=a**3*cos(2*a)  pred=a**3*cos(2*a)
       c1 [--]  gt=-2*a**3*sin(2*a) + 3*a**2*cos(2*a)  pred=-a**3*sin(2*a) + 3*a**2*cos(2*a)
       c2 [--]  gt=-4*a**3*cos(2*a) - 12*a**2*sin(2*a) + 6*a*cos(2*a)  pred=-a**3*cos(2*a) - 6*a**2*sin(2*a) + 6*a*cos(2*a)
       c3 [--]  gt=8*a**3*sin(2*a) - 36*a**2*cos(2*a) - 36*a*sin(2*a) + 6*cos(2*a)  pred=-